In [1]:
from transformer import ModelConfig, TransformerLM
model = TransformerLM.from_config(
  ModelConfig(
    vocab_size=50257,
    context_length=1024,
    d_model=768,
    n_layer=12,
    n_head=12,
    rope_base=10000.0,
    device='cpu'
  )
)

In [2]:
from transformers import AutoTokenizer
from trainutils import load_checkpoint
tokenizer = AutoTokenizer.from_pretrained("gpt2")
step = load_checkpoint("/FS1/models/mini-llm/checkpoint_interrupt_step_108518.pt", model, None, None)

In [5]:
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
prompts = [
    "The theory of evolution states that",
    "Once upon a time, there was a young",
    "The function takes an integer n and returns",
    "In 1969, the Apollo 11 mission",
    "Dear Sir or Madam,",
    'The capital of France is',
]
tokens = tokenizer(prompts, return_tensors="pt", padding=True).input_ids
model.eval()
answer_tokens = model.generate(tokens, max_new_tokens=20)
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
answer

' species sit with furry behavior: The brain advances virality that enhances their stellar arc. Anthropologists believe'

In [3]:
from datasets import load_from_disk
eval_dataset = load_from_disk("/FS1/datasets/openwebtext_tokenized_chunked_eval")

In [31]:
# randomly sample one example from eval_dataset['input_ids'][256:]
import random
random_sample = random.choice(eval_dataset['input_ids'][256:])[:128] # sample is a list of token ids
eval_sample = random.choice(eval_dataset['input_ids'][:256])[:128]
random_sample_txt = tokenizer.decode(random_sample, skip_special_tokens=True)
eval_sample_txt = tokenizer.decode(eval_sample, skip_special_tokens=True)
print("Random sample text:\n", random_sample_txt)
print("Eval sample text:\n", eval_sample_txt)

Random sample text:
  do an epicly bad job of acknowledging one another's work and checking our sources," he said.

Truong didn't expect Green to publicly address the situation on YouTube, but said it speaks to the kind of person he is.

"John was not obligated to do that, nor did he have to reply to any emails, posts, or offer to sell my artwork on his website," she said. "His act of generosity and kindness is something I will forever be grateful for. He publicly admitted his silly mistake and went above and beyond to correct the situation, which I believe is very admirable."

You can
Eval sample text:
 piercer, there is a twisted logic to the laws that govern it. Its passengers are what's left of humanity. The train’s failure spells the extinction of the human race, and it requires a cruel sense of order to run it.

Brutal and relentlessly dark

The film is brutal and relentlessly dark, but its most harrowing scenes aren’t characterized by unsettling acts of violence. Instead, Snowpi

In [32]:
import torch
random_sample_tensor = torch.tensor(random_sample).unsqueeze(0)
random_sample_output = model(random_sample_tensor, compute_loss=True)
print(f'Random sample loss: {random_sample_output.loss.item()}')

eval_sample_tensor = torch.tensor(eval_sample).unsqueeze(0)
eval_sample_output = model(eval_sample_tensor, compute_loss=True)
print(f'Eval sample loss: {eval_sample_output.loss.item()}')

Random sample loss: 3.690195083618164
Eval sample loss: 4.480434894561768
